<a href="https://colab.research.google.com/github/archaeogeek/colab/blob/main/excel_url_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Checking URLs

### Notes
We want to iterate through each sheet, and find anything that looks like a URL.

Our assumptions:
* a string is a URL if and only if it starts with http or https
* any strings that start with http or https but contain ; or ' ' should be split into multiple strings
* any URLs that end in a . or a ; should have that stripped

Then, we need to attempt to connect to each URL with a GET request and record whether it returns a success response or a failure.
We only want to check a URL once- the script should skip any that have already been checked.

Finally, we only want to return a list of the URLs reporting an error

### Usage

* Upload your xlsx file in the **files** section on the left.
* Click the three dots to the right of the file name and select **Copy path**
* In the code block below, replace the `file_path` with the path to your excel file
* For each block below in order, click the run buttons that appear when you mouse over the code blocks, to load the prerequisites and run the code. The results will be displayed in the output area below the final code block.

In [ ]:
# install required modules
!pip install openpyxl
!pip install google.colab.data_table

ERROR: Could not find a version that satisfies the requirement google.colab.data_table (from versions: none)
ERROR: No matching distribution found for google.colab.data_table


In [ ]:
# load the required modules
import openpyxl
import re
import requests
import warnings
import pandas as pd
from google.colab import data_table

### Code

In [ ]:
def find_urls_in_excel(file_path):
    # ignore warnings from openpyxl, we're not writing to the excel file
    warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
    # Load the workbook
    workbook = openpyxl.load_workbook(file_path)

    url_pattern = re.compile(r'(https?://[^\s]+)')

    url_list = []

    # Iterate through each sheet
    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        # Iterate through each row and column
        for row in sheet.iter_rows():
            for cell in row:
                if cell.value:
                    urls = url_pattern.findall(str(cell.value))
                    for url in urls:
                        sanitized_url = url.rstrip('.,!?;:)')
                        url_list.append({'sheet': sheet_name, 'cell': cell.coordinate, 'url': sanitized_url})
    return url_list

# check the status of each URL
def check_url_status(url):
    try:
        response = requests.get(url)
        return response.status_code
    except requests.exceptions.RequestException as e:
        return "Unspecified Error"


# put the path to the excel file here
file_path = '/content/2021 Angola Summary Data v2.xlsx'
urls = find_urls_in_excel(file_path)
checked_urls = []
url_data = []

# for each entry, check whether it has already been checked. If not, add to the
# checked_urls list. If it returns an error when the above function is called,
# add the details to the url_data list.
for entry in urls:
    if entry['url'] not in checked_urls:
        status_code = check_url_status(entry['url'])
        checked_urls.append(entry['url'])  # Mark this URL as checked
        # Only append the entry to the url_data list if the status code is not 200
        if status_code != 200:
            url_data.append({
                "Sheet": entry['sheet'],
                "Cell": entry['cell'],
                "URL": entry['url'],
                "Status": status_code
            })
        #print(url_data)

# Convert the url_data list into a dataframe formatted as a google colab data
# table for pretty printing
df = pd.DataFrame(url_data, columns=["Sheet", "Cell", "URL", "Status"])
data_table.enable_dataframe_formatter()
print('The following URLs returned an error. Note that duplicated URLs are not shown:')
df


The following URLs returned an error. Note that duplicated URLs are not shown:


,Sheet,Cell,URL,Status
0,Part 1 - About,E46,https://www.bna.ao/,Unspecified Error
1,Part 2 - Disclosure checklist,F48,https://www.bna.ao/#/pt/supervisao/prevencao-b...,Unspecified Error
2,Part 2 - Disclosure checklist,F52,https://www.endiama.co.ao/,406
3,Part 2 - Disclosure checklist,F60,https://mirempet.gov.ao/ao/documentos/relatori...,404


# Checking all formulae are in place

### Notes
We want to ensure that all the required formulae are in place in the sheet.

Our assumptions:
* we want to compare against a clean file, for which we use the empty template
* we need to normalise the formulae that we find to strip out unique row and column numbers, otherwise there will be a huge number of results, so we look for references within the formula to either `$COL$ROW` or `COLROW` and replace them with `$<COL>$<ROW>` or `<COL><ROW>` respectively
* We can't currently cope with array formulae, so we're silently ignoring them!



## Usage

Install the required packages

In [ ]:
!pip install gdown

Download a clean copy of the summary data template from google drive

In [ ]:
import gdown

file_id = '1_10xgkGZ41EsT6OE9aET54hsjk9hqKVn'
destination = 'en_eiti_summary_data_template_2.0.xlsx'  # The name of the file you want to save it as
gdown.download(f'https://drive.google.com/uc?id={file_id}', destination, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1_10xgkGZ41EsT6OE9aET54hsjk9hqKVn
To: /content/en_eiti_summary_data_template_2.0.xlsx
100%|██████████| 181k/181k [00:00<00:00, 54.2MB/s]


'en_eiti_summary_data_template_2.0.xlsx'

Then upload your summary data file for checking using the Files mechanism on the left, if you didn't already do that above

Adjust the `completed_wb` path in the code below to point at your file and then run the code.

The output will be displayed below the code block, grouped by sheet so that you can check your file against the template



## Code

In [ ]:
import openpyxl
import warnings
import re
import pandas as pd
from google.colab import data_table
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# Load the clean Summary Data Template you downloaded above
template_wb = '/content/en_eiti_summary_data_template_2.0.xlsx'

# Load the completed Summary Data file that you want to check
completed_wb = '/content/2019-2020 Tanzania Summary Data EN v2.xlsx'

# Regular expression pattern to match the pattern $COL$ROW
pattern_1 = re.compile(r'(?<=[$])[A-Z][$]\d+')

# Do a second pass to match the pattern COLROW
pattern_2 = re.compile(r'[A-Z]\d+')

# Function to substitute row numbers with <ROW>
def normalize_formula(formula):
    # Use re.sub to replace the matched row numbers with <ROW>
    try:
      return pattern_2.sub('<COL><ROW>', pattern_1.sub('<COL>$<ROW>', formula))
    except:
      return formula
    #return formula

# List to store distinct normalized formulas for the current sheet
distinct_formulas = []

def formula_exists(formula, distinct_formulas):
    for entry in distinct_formulas:
        if entry['normalized_formula'] == formula:
            return True
    return False

for work_book in [template_wb, completed_wb]:
  wb = openpyxl.load_workbook(work_book, data_only=False)
  # Process each sheet in the workbook
  for sheet_name in wb.sheetnames:
      ws = wb[sheet_name]

      # Iterate through the cells and check for formulas
      for row in ws.iter_rows():
          for cell in row:
              if cell.data_type == 'f':  # 'f' means the cell contains a formula
                  # Normalized formula for the current cell
                  formula = cell.value
                  if isinstance(formula, str):
                      normalized = normalize_formula(formula)
                  #else:
                  #   normalized = str(formula)
                  column, row = coordinate_from_string(cell.coordinate)
                  # If the normalized formula is not already in distinct_formulas, append it
                  if not formula_exists(normalized, distinct_formulas):
                    distinct_formulas.append({
                      "sheet": sheet_name,
                      "column": column,
                      "workbook": work_book,
                      "normalized_formula": normalized
                    })

'''Convert the distinct_formulas list into a dataframe, formatted as a google
colab data table for pretty printing, then split it into separate dataframes
filtered by sheet name'''

df = pd.DataFrame(distinct_formulas)
data_table.enable_dataframe_formatter()

sheets = df['sheet'].unique()
for sheet in sheets:
  print(f"\n{sheet}")
  display(df[df['sheet'] == sheet])



Part 1 - About


,sheet,column,workbook,normalized_formula
0,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,"=IFERROR(VLOOKUP($<COL>$<ROW>,Table1_Country_c..."
1,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,"=IFERROR(VLOOKUP($<COL>$<ROW>,Table1_Country_c..."
2,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,"=IFERROR(VLOOKUP($<COL>$<ROW>,Table1_Country_c..."
3,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(OR($<COL>$<ROW>=Lists!$<COL>$<ROW>,$<COL>$..."
4,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(OR($<COL>$<ROW>=Lists!$<COL>$<ROW>,$<COL>$..."
5,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(OR($<COL>$<ROW>=Lists!$<COL>$<ROW>,$<COL>$..."
6,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,=SUM(<COL><ROW>:<COL><ROW>)
7,Part 1 - About,E,/content/en_eiti_summary_data_template_2.0.xlsx,=COUNTIF('Part 2 - Disclosure checklist'!$D:$D...



Part 2 - Disclosure checklist


,sheet,column,workbook,normalized_formula
8,Part 2 - Disclosure checklist,F,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(<COL><ROW>=Lists!$<COL>$<ROW>,""< Input URL..."
9,Part 2 - Disclosure checklist,F,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(<COL><ROW>=Lists!$<COL>$<ROW>,""< Input URL..."
10,Part 2 - Disclosure checklist,D,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(OR(<COL><ROW>=Lists!$<COL>$<ROW>,),""< name..."
11,Part 2 - Disclosure checklist,F,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF(<COL><ROW>=""< name of the registry >"",""< I..."
12,Part 2 - Disclosure checklist,B,/content/en_eiti_summary_data_template_2.0.xlsx,"=LEFT(<COL><ROW>,SEARCH("","",<COL><ROW>))&"" value"""
13,Part 2 - Disclosure checklist,D,/content/en_eiti_summary_data_template_2.0.xlsx,=SUM('Part 5 - Company data'!<COL><ROW>/'Part ...
14,Part 2 - Disclosure checklist,B,/content/en_eiti_summary_data_template_2.0.xlsx,"=""Does the government disclose information on""..."
15,Part 2 - Disclosure checklist,D,/content/en_eiti_summary_data_template_2.0.xlsx,=IFERROR(IF(_xlfn.DAYS('Part 1 - About'!$<COL>...
30,Part 2 - Disclosure checklist,S,/content/2019-2020 Tanzania Summary Data EN v2...,"=PRODUCT(<COL><ROW>,<COL><ROW>)"
31,Part 2 - Disclosure checklist,B,/content/2019-2020 Tanzania Summary Data EN v2...,=<COL><ROW>/<COL><ROW>



Part 3 - Reporting entities


,sheet,column,workbook,normalized_formula
16,Part 3 - Reporting entities,E,/content/en_eiti_summary_data_template_2.0.xlsx,=SUMIF(Government_revenues_table[Government en...
17,Part 3 - Reporting entities,I,/content/en_eiti_summary_data_template_2.0.xlsx,"=(SUMIFS(Table10[Revenue value],Table10[Compan..."
32,Part 3 - Reporting entities,G,/content/2019-2020 Tanzania Summary Data EN v2...,=SUMIF(Government_revenues_table[Government en...
33,Part 3 - Reporting entities,G,/content/2019-2020 Tanzania Summary Data EN v2...,"=SUBTOTAL(109,<COL><ROW>:<COL><ROW>)"



Part 4 - Government revenues


,sheet,column,workbook,normalized_formula
18,Part 4 - Government revenues,N,/content/en_eiti_summary_data_template_2.0.xlsx,=Introduction!<COL><ROW>
19,Part 4 - Government revenues,B,/content/en_eiti_summary_data_template_2.0.xlsx,=IFERROR(VLOOKUP(Government_revenues_table[[#T...
20,Part 4 - Government revenues,C,/content/en_eiti_summary_data_template_2.0.xlsx,=IFERROR(VLOOKUP(Government_revenues_table[[#T...
21,Part 4 - Government revenues,D,/content/en_eiti_summary_data_template_2.0.xlsx,=IFERROR(VLOOKUP(Government_revenues_table[[#T...
22,Part 4 - Government revenues,E,/content/en_eiti_summary_data_template_2.0.xlsx,=IFERROR(VLOOKUP(Government_revenues_table[[#T...
23,Part 4 - Government revenues,J,/content/en_eiti_summary_data_template_2.0.xlsx,"=SUMIF(Government_revenues_table[Currency],""US..."
24,Part 4 - Government revenues,I,/content/en_eiti_summary_data_template_2.0.xlsx,"=""Total in ""&'Part 1 - About'!<COL><ROW>"
25,Part 4 - Government revenues,J,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF('Part 1 - About'!$<COL>$<ROW>=""USD"",0,SUMI..."
34,Part 4 - Government revenues,N,/content/2019-2020 Tanzania Summary Data EN v2...,=[1]Introduction!<COL><ROW>
35,Part 4 - Government revenues,B,/content/2019-2020 Tanzania Summary Data EN v2...,=IFERROR(VLOOKUP(Government_revenues_table[[#T...



Part 5 - Company data


,sheet,column,workbook,normalized_formula
26,Part 5 - Company data,B,/content/en_eiti_summary_data_template_2.0.xlsx,"=VLOOKUP(<COL><ROW>,Companies[],3,FALSE)"
27,Part 5 - Company data,J,/content/en_eiti_summary_data_template_2.0.xlsx,"=SUMIF(Table10[Reporting currency],""USD"",Table..."
28,Part 5 - Company data,H,/content/en_eiti_summary_data_template_2.0.xlsx,"=""Total in ""&'Part 1 - About'!$<COL>$<ROW>"
29,Part 5 - Company data,J,/content/en_eiti_summary_data_template_2.0.xlsx,"=IF('Part 1 - About'!$<COL>$<ROW>=""USD"",0,SUMI..."
41,Part 5 - Company data,B,/content/2019-2020 Tanzania Summary Data EN v2...,"=VLOOKUP(<COL><ROW>,[1]!Companies[#Data],3,FALSE)"


To do next:

*   Allow comparison with template spreadsheet
*   Display comparisons side by side in a table with a row for each worksheet



# Checking the number of columns in each table

## Notes

This module allows you to check that the number of columns in each data table defined in each sheet are correct, in comparison to the summary data template. If there are extra columns this can cause upload to the website to fail.

This module iterates through the summary data template, and for every sheet it prints out the name of every data table and the columns with their headers. If a column is hidden it appends an asterisk (*) to the column letter and column header.

## Usage

Run each module in turn, and the tables will be displayed at the end of the code section

## Code

In [ ]:
!pip install pandas openpyxl

In [ ]:
import gdown

file_id = '1_10xgkGZ41EsT6OE9aET54hsjk9hqKVn'
destination = 'en_eiti_summary_data_template_2.0.xlsx'  # The name of the file you want to save it as
gdown.download(f'https://drive.google.com/uc?id={file_id}', destination, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1_10xgkGZ41EsT6OE9aET54hsjk9hqKVn
To: /content/en_eiti_summary_data_template_2.0.xlsx
100%|██████████| 181k/181k [00:00<00:00, 45.4MB/s]


'en_eiti_summary_data_template_2.0.xlsx'

In [ ]:
import openpyxl
import pandas as pd
from openpyxl.utils import get_column_letter, column_index_from_string
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# Load the workbook and worksheet
file_path = '/content/en_eiti_summary_data_template_2.0.xlsx'
wb = openpyxl.load_workbook(file_path, data_only=False)

# Process each sheet in the workbook
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Access tables in the worksheet
    tables = ws.tables  # Dictionary of tables

    # Loop through each table to get the name and number of columns
    for table_name, table_range in tables.items():
        print(f"\nSheet: {sheet_name} | Table: {table_name}")

        # Split the range to get starting and ending cell coordinates
        start_cell, end_cell = table_range.split(":")

        # Extract column letters and row numbers from start and end cells
        start_col = ''.join([i for i in start_cell if i.isalpha()])  # Extract column letter (e.g., 'A')
        start_row = ''.join([i for i in start_cell if i.isdigit()])  # Extract row number (e.g., '1')

        end_col = ''.join([i for i in end_cell if i.isalpha()])      # Extract end column letter
        end_row = ''.join([i for i in end_cell if i.isdigit()])      # Extract end row number


        # Convert column letters to indices
        start_col_idx = column_index_from_string(start_col)
        end_col_idx = column_index_from_string(end_col)

        # Create a list to hold the column letters and headers
        column_info = []

        # Loop through each column in the range and extract column names (header)
        for col_idx in range(start_col_idx, end_col_idx + 1):
            col_letter = get_column_letter(col_idx)
            # Get the header in the first row of the table (based on the start_row)
            cell_value = ws[f'{col_letter}{start_row}'].value

            # Check if the column is hidden and flag it with an asterisk if hidden
            if ws.column_dimensions[col_letter].hidden:
                cell_value = f"{cell_value} *" if cell_value else "*"
                col_letter = f"{col_letter} *" if col_letter else "*"

            column_info.append([col_letter, cell_value])


        # Convert the column info to a pandas DataFrame for display
        df_columns = pd.DataFrame(column_info, columns=['Column Letter', 'Column Header'])

        # Display the DataFrame for the current table
        display(df_columns)



Sheet: Part 3 - Reporting entities | Table: Companies


,Column Letter,Column Header
0,B,Full company name
1,C,Company type
2,D,Company ID number
3,E,Sector
4,F,Commodities (comma-seperated)
5,G,Stock exchange listing or company website
6,H,"Audited financial statement (or balance sheet,..."
7,I,Payments to Governments Report



Sheet: Part 3 - Reporting entities | Table: Government_agencies


,Column Letter,Column Header
0,B,Full name of agency
1,C,Agency type
2,D,ID number (if applicable)
3,E,Total reported



Sheet: Part 3 - Reporting entities | Table: Companies15


,Column Letter,Column Header
0,B,Full project name
1,C,"Legal agreement reference number(s): contract,..."
2,D,"Affiliated companies, start with Operator"
3,E,Commodities (one commodity/row)
4,F,Status
5,G,Production (volume)
6,H,Unit
7,I,Production (value)
8,J,Currency


... (18 outputs left out)


Sheet: Lists | Table: Table12


,Column Letter,Column Header
0,AC,Project phases



Sheet: Lists | Table: Government_entity_type


,Column Letter,Column Header
0,AE,< Agency type >
